In [ ]:
from pathlib import Path
import copy
import gc
import hashlib
import json
import math
import random
import unicodedata
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from torch.utils.data import Dataset, DataLoader

DATA_DIR = Path("data")
OUTPUT_DIR = Path("results/leave_one_drug_pair_out_seed42")
SEED = 42
ALPHA = 0.5
VAL_PAIR_FRACTION = 0.125
MAX_EPOCHS = 100
BATCH_SIZE = 8
INITIAL_LR = 5e-5
EARLY_STOP_PATIENCE = 10
MIN_DELTA = 1e-6
RESUME = True
FOLD_IDS = None
SAVE_MODELS = False
TARGET_FILE = "processed_combination_response_r070_clean_with_qc.csv"
DRUG_FEATURE_FILE = "Graphlet_features_6_standardized.csv"
CELL_FEATURE_FILE = "cell_features_977d.csv"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

IMPLEMENTATION_HASH = "e4d4d81328e1ae662a835d126af97b6b45b456c3039ad3f5190527c160739160"


In [ ]:
def normalize_name(value):
    if pd.isna(value):
        raise ValueError("Missing drug/cell name")
    value = unicodedata.normalize("NFKC", str(value)).strip().upper()
    return value.translate(str.maketrans({"–": "-", "—": "-", "−": "-"}))


def load_data(data_dir=DATA_DIR):
    paths = [data_dir / f for f in [TARGET_FILE, DRUG_FEATURE_FILE, CELL_FEATURE_FILE]]
    for path in paths:
        if not path.is_file():
            raise FileNotFoundError(f"Missing {path.resolve()}; set DATA_DIR to your data folder.")
    raw, drugs, cells = [pd.read_csv(p, encoding="utf-8-sig") for p in paths]
    required = ["drugA_name", "drugB_name", "cell_line", "drugA_conc", "drugB_conc",
                "target", "single_resp_1", "single_resp_2"]
    if not set(required).issubset(raw.columns):
        raise ValueError(f"Missing response columns: {set(required) - set(raw.columns)}")
    for col in ["drugA_name", "drugB_name", "cell_line"]:
        raw[col] = raw[col].map(normalize_name)
    drugs["name"] = drugs["name"].map(normalize_name)
    cells["Cell_Line"] = cells["Cell_Line"].map(normalize_name)
    for frame, key in [(drugs, "name"), (cells, "Cell_Line")]:
        if frame[key].duplicated().any():
            raise ValueError(f"Duplicate normalized feature names: {frame.loc[frame[key].duplicated(), key].tolist()}")
    drugs = drugs.set_index("name").drop(columns=["smile", "mol"], errors="ignore")
    cells = cells.set_index("Cell_Line")
    drugs = drugs.apply(pd.to_numeric, errors="raise").astype(np.float32)
    cells = cells.apply(pd.to_numeric, errors="raise").astype(np.float32)
    if drugs.shape[1] != 6890 or cells.shape[1] != 977:
        raise ValueError(f"Expected drug/cell dimensions 6890/977, got {drugs.shape[1]}/{cells.shape[1]}")
    if not np.isfinite(drugs.to_numpy()).all() or not np.isfinite(cells.to_numpy()).all():
        raise ValueError("Nonfinite feature values")
    numeric = ["drugA_conc", "drugB_conc", "target", "single_resp_1", "single_resp_2"]
    raw[numeric] = raw[numeric].apply(pd.to_numeric, errors="raise")
    if not np.isfinite(raw[numeric].to_numpy()).all():
        raise ValueError("Nonfinite response/dose values: fix source data before CV")
    if (raw[["drugA_conc", "drugB_conc"]] < 0).any().any():
        raise ValueError("Negative concentrations")
    if "combo_xx0" in raw and not np.allclose(raw.target, 1 - raw.combo_xx0, atol=1e-6):
        raise ValueError("target must be inhibition = 1 - combo_xx0")
    mask = raw.drugA_name.isin(drugs.index) & raw.drugB_name.isin(drugs.index) & raw.cell_line.isin(cells.index)
    df = raw.loc[mask, required].copy()
    df["row_id"] = np.flatnonzero(mask)
    swap = df.drugA_name > df.drugB_name
    for a, b in [("drugA_name", "drugB_name"), ("drugA_conc", "drugB_conc"), ("single_resp_1", "single_resp_2")]:
        df.loc[swap, [a, b]] = df.loc[swap, [b, a]].to_numpy()
    df = df.reset_index(drop=True)
    df["drug_pair_id"] = [json.dumps([a, b], ensure_ascii=False) for a, b in zip(df.drugA_name, df.drugB_name)]
    print(f"Input {len(raw)}; missing-feature exclusions {int((~mask).sum())}; retained {len(df)}")
    print(f"Drug pairs {df.drug_pair_id.nunique()}; drugs {len(set(df.drugA_name) | set(df.drugB_name))}; cells {df.cell_line.nunique()}")
    if df.drug_pair_id.nunique() < 3:
        raise ValueError("At least three drug pairs are required")
    return df, drugs, cells


def make_split(df, test_pair, seed=SEED):
    pairs = np.array(sorted(df.drug_pair_id.unique()))
    remaining = pairs[pairs != test_pair]
    if test_pair not in pairs or len(remaining) < 2:
        raise ValueError("Invalid held-out drug pair")
    n_val = min(len(remaining) - 1, max(1, math.ceil(len(remaining) * VAL_PAIR_FRACTION)))
    val_pairs = np.random.default_rng(seed).permutation(remaining)[:n_val]
    test = df.loc[df.drug_pair_id.eq(test_pair)].copy().reset_index(drop=True)
    val = df.loc[df.drug_pair_id.isin(val_pairs)].copy().reset_index(drop=True)
    train = df.loc[~df.drug_pair_id.isin([test_pair, *val_pairs])].copy().reset_index(drop=True)
    subsets = [train, val, test]
    for i, left in enumerate(subsets):
        assert len(left) > 0 and left.row_id.is_unique
        for right in subsets[i + 1:]:
            assert set(left.drug_pair_id).isdisjoint(right.drug_pair_id)
            assert set(left.row_id).isdisjoint(right.row_id)
    assert test.drug_pair_id.nunique() == 1
    assert sum(map(len, subsets)) == len(df)
    assert set().union(*(set(x.row_id) for x in subsets)) == set(df.row_id)
    return train, val, test


def audit_all_splits(df):
    records = []
    coverage = pd.Series(0, index=df.row_id, dtype=np.int32)
    for fold_id, pair in enumerate(sorted(df.drug_pair_id.unique())):
        tr, va, te = make_split(df, pair)
        coverage.loc[te.row_id] += 1
        records.append(dict(fold_id=fold_id, test_pair=pair, train_samples=len(tr),
                            val_samples=len(va), test_samples=len(te),
                            train_pairs=tr.drug_pair_id.nunique(), val_pairs=va.drug_pair_id.nunique(), test_pairs=1,
                            train_val_overlap=0, train_test_overlap=0, val_test_overlap=0))
    assert coverage.eq(1).all(), "Every retained row must be tested exactly once"
    return pd.DataFrame(records)


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class DrugCombinationDataset(Dataset):
    def __init__(self, data_df, drug_features, cell_features):
        self.data = data_df.reset_index(drop=True)

        self.drug_names = drug_features.index.astype(str).tolist()
        self.cell_names = cell_features.index.astype(str).tolist()
        self.drug_to_index = {name: i for i, name in enumerate(self.drug_names)}
        self.cell_to_index = {name: i for i, name in enumerate(self.cell_names)}

        self.drug_matrix = np.ascontiguousarray(drug_features.to_numpy(dtype=np.float32))
        self.cell_matrix = np.ascontiguousarray(cell_features.to_numpy(dtype=np.float32))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        drug_a = row["drugA_name"]
        drug_b = row["drugB_name"]
        cell_line = row["cell_line"]

        return {
            "drugA_feat": torch.from_numpy(self.drug_matrix[self.drug_to_index[drug_a]]),
            "drugB_feat": torch.from_numpy(self.drug_matrix[self.drug_to_index[drug_b]]),
            "cell_feat": torch.from_numpy(self.cell_matrix[self.cell_to_index[cell_line]]),
            "target": torch.tensor(row["target"], dtype=torch.float32),
            "drugA_conc": torch.tensor(row["drugA_conc"], dtype=torch.float32),
            "drugB_conc": torch.tensor(row["drugB_conc"], dtype=torch.float32),
            "single_resp_1": torch.tensor(row["single_resp_1"], dtype=torch.float32),
            "single_resp_2": torch.tensor(row["single_resp_2"], dtype=torch.float32),
            "index": idx,
            "drugA_name": drug_a,
            "drugB_name": drug_b,
            "cell_line": cell_line,
        }

class DrugCombinationModel(nn.Module):
    def __init__(self, drug_feat_dim=1000, cell_feat_dim=977):
        super(DrugCombinationModel, self).__init__()

        self.drug_encoder = nn.Sequential(
            nn.Linear(drug_feat_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
        )

        self.cell_encoder = nn.Sequential(
            nn.Linear(cell_feat_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
        )


        self.theta1_net = nn.Sequential(
            nn.Linear(256 + 256 + 1, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

        self.theta2_net = nn.Sequential(
            nn.Linear(256 + 256 + 1, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )


        self.epsilon_net = nn.Sequential(
            nn.Linear(256 * 3 + 256 + 2, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 1),
        )


        self.dose_encoder = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
        )


        hpb_input_dim = 256 + 256 + 256 + 32 + 32
        self.predictor_direct = nn.Sequential(
            nn.Linear(hpb_input_dim, 2048),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(
        self,
        drugA_feat,
        drugB_feat,
        cell_feat,
        combo_feat,
        drugA_conc,
        drugB_conc,
    ):
        encoded_drugA = self.drug_encoder(drugA_feat)
        encoded_drugB = self.drug_encoder(drugB_feat)
        encoded_cell = self.cell_encoder(cell_feat)

        theta1_input = torch.cat(
            [encoded_drugA, encoded_cell, drugA_conc.unsqueeze(1)], dim=1
        )
        theta2_input = torch.cat(
            [encoded_drugB, encoded_cell, drugB_conc.unsqueeze(1)], dim=1
        )
        theta1_raw = self.theta1_net(theta1_input)
        theta2_raw = self.theta2_net(theta2_input)

        drug_pair_sum = encoded_drugA + encoded_drugB
        drug_pair_product = encoded_drugA * encoded_drugB
        drug_pair_abs_difference = torch.abs(encoded_drugA - encoded_drugB)
        epsilon_input = torch.cat(
            [
                drug_pair_sum,
                drug_pair_product,
                drug_pair_abs_difference,
                encoded_cell,
                drugA_conc.unsqueeze(1),
                drugB_conc.unsqueeze(1),
            ],
            dim=1,
        )
        epsilon = self.epsilon_net(epsilon_input)

        theta12_sum = theta1_raw + theta2_raw
        theta1 = theta1_raw / theta12_sum
        theta2 = theta2_raw / theta12_sum

        encoded_doseA = self.dose_encoder(drugA_conc.unsqueeze(1))
        encoded_doseB = self.dose_encoder(drugB_conc.unsqueeze(1))

        combined_features = torch.cat(
            [
                encoded_drugA,
                encoded_drugB,
                encoded_cell,
                encoded_doseA,
                encoded_doseB,
            ],
            dim=1,
        )
        p_direct = self.predictor_direct(combined_features)

        return (
            theta1.squeeze(-1),
            theta2.squeeze(-1),
            epsilon.squeeze(-1),
            p_direct.squeeze(-1),
        )

In [ ]:
def concordance_index_continuous(targets, predictions):

    targets = np.asarray(targets, dtype=np.float64)
    predictions = np.asarray(predictions, dtype=np.float64)
    mask = np.isfinite(targets) & np.isfinite(predictions)
    targets = targets[mask]
    predictions = predictions[mask]

    if len(targets) < 2:
        return np.nan

    order = np.argsort(targets, kind="mergesort")
    targets = targets[order]
    predictions = predictions[order]
    _, pred_ranks = np.unique(predictions, return_inverse=True)
    pred_ranks = pred_ranks + 1
    tree = np.zeros(int(pred_ranks.max()) + 1, dtype=np.int64)

    def update(index):
        while index < len(tree):
            tree[index] += 1
            index += index & -index

    def query(index):
        count = 0
        while index > 0:
            count += tree[index]
            index -= index & -index
        return count

    concordant = 0.0
    comparable = 0
    previous_count = 0
    start = 0
    while start < len(targets):
        end = start + 1
        while end < len(targets) and targets[end] == targets[start]:
            end += 1

        for rank in pred_ranks[start:end]:
            less = query(int(rank) - 1)
            less_or_equal = query(int(rank))
            equal = less_or_equal - less
            concordant += less + 0.5 * equal
            comparable += previous_count

        for rank in pred_ranks[start:end]:
            update(int(rank))
        previous_count += end - start
        start = end

    return concordant / comparable if comparable else np.nan

def forward_batch(model, batch):
    keys = ["drugA_feat", "drugB_feat", "cell_feat", "drugA_conc", "drugB_conc", "target", "single_resp_1", "single_resp_2"]
    a, b, c, da, db, y, ra, rb = [batch[k].to(device) for k in keys]
    w1, w2, eps, hpb = model(a, b, c, None, da, db)
    idb = w1 * ra + w2 * rb + eps
    return y, hpb, idb, w1, w2, eps


def joint_loss(y, hpb, idb):
    return ALPHA * nn.functional.mse_loss(idb, y) + (1 - ALPHA) * nn.functional.mse_loss(hpb, y)


def make_loader(df, drugs, cells, shuffle=False):
    return DataLoader(DrugCombinationDataset(df, drugs, cells), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=0,
                      generator=torch.Generator().manual_seed(SEED))


@torch.no_grad()
def validation_mse(model, loader):
    model.eval()
    squared_error = 0.0
    n = 0
    for batch in loader:
        y, hpb, *_ = forward_batch(model, batch)
        squared_error += torch.sum((hpb - y) ** 2).item()
        n += y.numel()
    score = squared_error / n
    if not np.isfinite(score):
        raise FloatingPointError("Nonfinite validation HPB MSE")
    return score


def fit_fold(train_loader, val_loader, drug_dim, cell_dim):
    set_seed(SEED)
    model = DrugCombinationModel(drug_dim, cell_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=INITIAL_LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-7)
    best, best_epoch, stale, best_state = float("inf"), 0, 0, None
    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        loss_sum = 0.0
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            y, hpb, idb, *_ = forward_batch(model, batch)
            loss = joint_loss(y, hpb, idb)
            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite training loss")
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * y.numel()
        score = validation_mse(model, val_loader)
        scheduler.step(score)
        history.append(dict(epoch=epoch, train_joint_mse=loss_sum / len(train_loader.dataset),
                            val_hpb_mse=score, lr=optimizer.param_groups[0]["lr"]))
        if score < best - MIN_DELTA:
            best, best_epoch, stale = score, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
        print(f"  epoch {epoch}: joint loss {history[-1]['train_joint_mse']:.6f}; val HPB MSE {score:.6f}", flush=True)
        if stale >= EARLY_STOP_PATIENCE:
            break
    if best_state is None:
        raise RuntimeError("No valid checkpoint")
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch


@torch.no_grad()
def predict_fold(model, loader, test_df):
    model.eval()
    records = []
    for batch in loader:
        y, hpb, idb, w1, w2, eps = forward_batch(model, batch)
        indices = batch["index"].numpy()
        out = test_df.iloc[indices].copy()
        for name, value in [("prediction", hpb), ("prediction_idb", idb), ("theta1", w1), ("theta2", w2), ("epsilon", eps)]:
            out[name] = value.cpu().numpy()
        records.append(out)
    result = pd.concat(records, ignore_index=True)
    assert len(result) == len(test_df) and result.row_id.is_unique
    assert np.allclose(result.theta1 + result.theta2, 1, atol=1e-6)
    return result


def regression_metrics(y, pred):
    y, pred = np.asarray(y, dtype=float), np.asarray(pred, dtype=float)
    if len(y) != len(pred) or not len(y) or not np.isfinite(y).all() or not np.isfinite(pred).all():
        raise ValueError("Invalid targets or predictions")
    mse = mean_squared_error(y, pred)
    return dict(mse=float(mse), rmse=float(np.sqrt(mse)), mae=float(mean_absolute_error(y, pred)),
                pcc=float(np.corrcoef(y, pred)[0, 1]) if len(y) > 1 and np.std(y) > 0 and np.std(pred) > 0 else np.nan,
                r2=float(r2_score(y, pred)) if len(y) > 1 and np.std(y) > 0 else np.nan,
                c_index=float(concordance_index_continuous(y, pred)))


In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def run_cv():
    df, drugs, cells = load_data(DATA_DIR)
    audit = audit_all_splits(df)
    config = dict(protocol="LOPO_joint_loss_HPB_canonical_order_v1", seed=SEED, alpha=ALPHA,
                  val_fraction=VAL_PAIR_FRACTION, epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
                  lr=INITIAL_LR, patience=EARLY_STOP_PATIENCE, min_delta=MIN_DELTA,
                  source_hash=IMPLEMENTATION_HASH,
                  data={name: sha256_file(DATA_DIR / name) for name in [TARGET_FILE, DRUG_FEATURE_FILE, CELL_FEATURE_FILE]})
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    config_path = OUTPUT_DIR / "run_config.json"
    if config_path.exists():
        if json.loads(config_path.read_text()) != config:
            raise ValueError("Data/code/settings differ from saved run; use a new OUTPUT_DIR")
    elif any(OUTPUT_DIR.iterdir()):
        raise ValueError("Nonempty output folder has no run configuration; use a new OUTPUT_DIR")
    config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
    audit.to_csv(OUTPUT_DIR / "all_folds_split_audit.csv", index=False)
    pairs = audit.test_pair.tolist()
    selected = list(range(len(pairs))) if FOLD_IDS is None else list(FOLD_IDS)
    if len(selected) != len(set(selected)) or any(not isinstance(i, int) or i < 0 or i >= len(pairs) for i in selected):
        raise ValueError("FOLD_IDS must contain distinct valid integer fold IDs")
    print(f"{len(pairs)} outer folds; {len(selected)} scheduled; train/validation pairs are disjoint.")
    for fold_id in selected:
        fold_dir = OUTPUT_DIR / f"fold_{fold_id:04d}"
        fold_dir.mkdir(exist_ok=True)
        done = fold_dir / "complete.json"
        if RESUME and done.exists():
            info = json.loads(done.read_text())
            if info["prediction_sha256"] != sha256_file(fold_dir / "predictions.csv"):
                raise ValueError(f"Corrupt predictions in {fold_dir}")
            continue
        done.unlink(missing_ok=True)
        tr, va, te = make_split(df, pairs[fold_id])
        split_info = {"test_pair": pairs[fold_id], "train_pairs": sorted(tr.drug_pair_id.unique()), "val_pairs": sorted(va.drug_pair_id.unique())}
        (fold_dir / "split.json").write_text(json.dumps(split_info, indent=2), encoding="utf-8")
        print(f"Fold {fold_id + 1}/{len(pairs)}: {pairs[fold_id]}", flush=True)
        train_loader = make_loader(tr, drugs, cells, True)
        val_loader = make_loader(va, drugs, cells)
        model, history, best_epoch = fit_fold(train_loader, val_loader, drugs.shape[1], cells.shape[1])
        predictions = predict_fold(model, make_loader(te, drugs, cells), te)
        predictions["fold_id"] = fold_id
        predictions.to_csv(fold_dir / "predictions.csv", index=False)
        history.to_csv(fold_dir / "history.csv", index=False)
        if SAVE_MODELS:
            torch.save(model.state_dict(), fold_dir / "best_model.pth")
        record = dict(fold_id=fold_id, best_epoch=best_epoch,
                      prediction_sha256=sha256_file(fold_dir / "predictions.csv"))
        tmp = fold_dir / "complete.tmp"
        tmp.write_text(json.dumps(record), encoding="utf-8")
        tmp.replace(done)
        print(regression_metrics(predictions.target, predictions.prediction), flush=True)
        del model, train_loader, val_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    predictions, metrics = [], []
    for fold_id, pair in enumerate(pairs):
        fold_dir = OUTPUT_DIR / f"fold_{fold_id:04d}"
        if not (fold_dir / "complete.json").exists():
            continue
        info = json.loads((fold_dir / "complete.json").read_text())
        if info["prediction_sha256"] != sha256_file(fold_dir / "predictions.csv"):
            raise ValueError(f"Corrupt predictions in {fold_dir}")
        pred = pd.read_csv(fold_dir / "predictions.csv")
        expected = df.loc[df.drug_pair_id.eq(pair)].set_index("row_id")
        assert pred.row_id.is_unique and set(pred.row_id) == set(expected.index)
        assert pred.drug_pair_id.eq(pair).all() and pred.fold_id.eq(fold_id).all()
        assert np.allclose(pred.target, expected.loc[pred.row_id, "target"], atol=1e-7)
        predictions.append(pred)
        metrics.append(dict(fold_id=fold_id, test_pair=pair, n_test=len(pred), best_epoch=info["best_epoch"],
                            **regression_metrics(pred.target, pred.prediction)))
    if not predictions:
        print("No completed folds yet")
        return
    oof = pd.concat(predictions, ignore_index=True).sort_values("row_id")
    assert oof.row_id.is_unique
    complete = len(predictions) == len(pairs)
    prefix = "complete" if complete else "partial"
    if complete:
        assert set(oof.row_id) == set(df.row_id)
    fold_metrics = pd.DataFrame(metrics)
    oof.to_csv(OUTPUT_DIR / f"{prefix}_oof_predictions.csv", index=False)
    fold_metrics.to_csv(OUTPUT_DIR / f"{prefix}_fold_metrics.csv", index=False)
    pooled = regression_metrics(oof.target, oof.prediction)
    pd.DataFrame([dict(status=prefix, completed_folds=len(predictions), total_folds=len(pairs), n_samples=len(oof), **pooled)]).to_csv(
        OUTPUT_DIR / f"{prefix}_pooled_metrics.csv", index=False)
    columns = ["mse", "rmse", "mae", "pcc", "r2", "c_index"]
    macro = fold_metrics[columns].agg(["mean", "std", "count"]).T.rename(columns={"std": "sample_sd", "count": "valid_folds"})
    macro.to_csv(OUTPUT_DIR / f"{prefix}_macro_metrics.csv", index_label="metric")
    print(f"Status: {prefix}; completed folds {len(predictions)}/{len(pairs)}")
    print("Pooled out-of-fold metrics (HPB):")
    print(pd.Series(pooled).to_string(float_format=lambda x: f"{x:.4f}"))
    print("Per-pair macro mean / sample SD / number of valid folds:")
    print(macro.to_string(float_format=lambda x: f"{x:.4f}"))


In [ ]:
run_cv()